# Curso de Férias: Biopython


- Nosso conteúdo
  - [Canva](https://www.canva.com/design/DAG_osc-YGg/sWj0-_oCf7b9Y76z4LHICQ/view)
  - [GitHub](https://github.com/jvfd3/Curso-de-Ferias_BioPython)
- **Conteúdo base:**
  - [Documentação do biopython](https://biopython.org/wiki/SeqIO)
  - [Tutorial e Cookbook](https://biopython.org/docs/latest/Tutorial/index.html)
- **Conteúdo adicional no GitHub:**
  - [peterjc/biopython_workshop](https://github.com/peterjc/biopython_workshop/blob/master/reading_sequence_files/README.rst)
  - [tiagoantao/biopython-notebook](https://github.com/tiagoantao/biopython-notebook)
  - [peterjc/biopython_workshop](https://github.com/peterjc/biopython_workshop)


## Conteúdo


- **Conceitos**:
  - Bio.Seq
  - Bio.SeqUtils
  - Bio.SeqIO
  - Bio.SeqRecord
  - Bio.SeqFeature
  - Bio.Align

- **Segmentos**:
  - Preparo do material
    - Instalando Biopython
    - Obtendo FASTA do NCBI
    - Obtendo arquivo do ExPASy
    - Obtendo arquivos diretamente da internet
  - Sequências
    - Objeto Seq
      - Métodos do Seq
    - Tratamento de Erros
    - Seq Avançado
      - Objetos MutableSeq (sequências editáveis)
    - Leitura e Escrita de Sequências: SeqIO
      - Lendo Arquivos
        - Lendo arquivo FASTA
        - Lendo arquivo GenBank
        - Lendo arquivos compactados (gzip)
        - Carregando o arquivo para a memória
        - Percorrendo registros
      - Escrevendo/Convertendo arquivos
    - Seq Record
      - Fatiando um SeqRecord
      - Escrevendo arquivos
    - Seq Feature
      - Exemplo real do Seq Feature em um arquivo GenBank
  - Bio Align
    - Criando um Alignment a partir de sequências já alinhadas
    - Comparando alinhamentos
    - Contagem de identidades, mismatches, lacunas e estatísticas
    - Alinhamento Múltiplo


## Preparo do material


Antes de começarmos, primeiro precisamos instalar o Biopython. E, em seguida, baixaremos os arquivos de sequência do NCBI para trabalharmos com eles durante o tutorial.


### Instalando Biopython


Para executarmos os códigos deste tutorial, precisamos instalar o Biopython. Você pode fazer isso usando pip, que é o gerenciador de pacotes do Python.

Ao utilizá-lo em um ambiente Jupyter Notebook, pode-se usar o comando mágico `%pip` para garantir que a instalação seja feita no ambiente correto. Além disso, podemos usar algumas opções (flags):

- `-q` ou `--quiet`: para reduzir a quantidade de saída durante a instalação, tornando o processo mais silencioso.
- `==1.86`: para especificar a versão do Biopython que queremos instalar

Como estamos usando o IPython Notebook no Google Colab, temos à disposição os "comandos mágicos" iniciados por `%` que permitem executar comandos do sistema diretamente do notebook. E assim podemos instalar o Biopython usando o seguinte comando.


In [1]:
''' Instalando Biopython '''

%pip -q install biopython==1.86

Note: you may need to restart the kernel to use updated packages.


### Obtendo FASTA do NCBI


para trabalharmos com o Biopython, vamos precisar de um arquivo de sequência, um dos formatos mais comuns é o FASTA, que é um formato de texto simples para representar sequências biológicas. O Biopython tem uma funcionalidade integrada para baixar sequências do NCBI usando o módulo `Entrez`.


In [2]:
from Bio import Entrez  # Importando módulo para acessar o NCBI

A função `get_fasta_from_ncbi` abaixo exemplifica como fazer isso.


In [3]:
''' Obter arquivos '''


def save_from_entrez_to_file(accession: str, extension: str = 'fasta', email: str = 'example@mail.com', db: str = 'nuccore') -> None:
    ''' Baixa uma sequência FASTA do NCBI dado um
        número de acesso e salva em um arquivo '''
    Entrez.email = email
    FILENAME = f'{accession}.{extension}'

    received_accession = Entrez.efetch(
        db=db,
        id=accession,
        rettype=extension,
        retmode='text'
    )
    with received_accession as handle:
        # Aqui, ao invés de salvar o arquivo, poderíamos processar o conteúdo diretamente
        with open(FILENAME, 'w') as fasta_file:
            fasta_file.write(handle.read())


# Baixando arquivos necessários para o curso
EMAIL = 'seu_email@exemplo.com'  # Substitua pelo seu email real
save_from_entrez_to_file('NC_045512.2', 'fasta', EMAIL)
# save_from_entrez_to_file('NC_005816', 'gb', EMAIL)

### Obtendo arquivo do ExPASy


In [4]:
from Bio import ExPASy

In [5]:
''' Acessando o SwissProt pela internet (mesma lógica do anterior) Só que aqui usa o Bio.ExPASy '''


def save_from_expasy_to_file(accession: str, extension: str = 'txt') -> None:
    ''' Baixa uma sequência do SwissProt dado um
        número de acesso e salva em um arquivo '''

    FILENAME = f'{accession}.{extension}'
    with ExPASy.get_sprot_raw(accession) as handle:
        with open(FILENAME, 'w') as file:
            # Decodificando bytes para string
            data = handle.read().decode('utf-8')
            file.write(data)

# save_from_expasys_to_file('O23729')

### Obtendo arquivos diretamente da internet


Anteriormente vimos como baixar arquivos do NCBI usando o módulo `Entrez`. No entanto, às vezes, os arquivos que queremos usar já estão disponíveis na internet, e podemos baixá-los diretamente usando a biblioteca `urllib.request` do Python.

No exemplo abaixo, mostramos como usar a função `urlretrieve` para baixar um arquivo FASTA diretamente de um URL.


In [6]:
from urllib.request import urlretrieve

In [7]:

''' Download de arquivos usando Python

Aqui baixaremos alguns dos arquivos que usaremos durante o curso
'''

files_to_download = {
    'NC_005816.fna': 'https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/NC_005816.fna',
    'ls_orchid.fasta': 'https://raw.githubusercontent.com/biopython/biopython/refs/heads/master/Doc/examples/ls_orchid.fasta',
    'ls_orchid.gbk': 'https://raw.githubusercontent.com/biopython/biopython/refs/heads/master/Doc/examples/ls_orchid.gbk',
    'NC_005816.gb': 'https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/NC_005816.gb',
    'ls_orchid.gbk.gz': 'https://raw.githubusercontent.com/biopython/biopython/refs/heads/master/Doc/examples/ls_orchid.gbk.gz',
    'ucsc_mm9_chr10.maf': 'https://raw.githubusercontent.com/biopython/biopython/refs/heads/master/Tests/MAF/ucsc_mm9_chr10.maf',
}

for filename, url in files_to_download.items():
    urlretrieve(url, filename)

## Sequências


Sequências biológicas são representações de nucleotídeos (DNA/RNA) ou aminoácidos (proteínas). O Biopython oferece a classe `Seq` para manipular essas sequências de forma eficiente e intuitiva. A classe `Seq` é parte do módulo `Bio.Seq` e fornece métodos para realizar operações comuns em sequências, como transcrição, tradução, complementação, entre outras.


### Objeto Seq


O objeto `Seq` é a classe central para representar sequências biológicas no Biopython. Ele é imutável, o que significa que uma vez criado, seu conteúdo não pode ser alterado. Isso garante a integridade das sequências e permite otimizações internas. O `Seq` apresenta algumas operações similares às strings como fatiamento, concatenação, mas com funcionalidades adicionais para biologia molecular como transcrição, tradução, complementação, etc.


In [8]:
from Bio.Seq import Seq

In [9]:
''' Manipulando Sequências '''

my_seq_1 = Seq('GATCG')

print('Índice|Letra')
for index, letter in enumerate(my_seq_1):
    print(f'{index}|{letter}')

print('Tamanho da sequência:', len(my_seq_1))
print('Quantidade de bases A:', my_seq_1.count('A'))

print('Primeira base:', my_seq_1[0])
print('Terceira base:', my_seq_1[2])
print('Última base:', my_seq_1[-1])

Índice|Letra
0|G
1|A
2|T
3|C
4|G
Tamanho da sequência: 5
Quantidade de bases A: 1
Primeira base: G
Terceira base: T
Última base: G


In [10]:
''' Fatiamento de sequências '''

my_seq_2 = Seq('GATCGATGGGCCTATATAGGATCGAAAATCGC')
print('Fatiamento simples (posição 4 a 11):', my_seq_2[4:12])
print('Segunda base de cada códon:', my_seq_2[0::3])
print('Terceira base de cada códon:', my_seq_2[1::3])
print('Terceira base de cada códon:', my_seq_2[0::3])
print('Sequência invertida:', my_seq_2[::-1])

Fatiamento simples (posição 4 a 11): GATGGGCC
Segunda base de cada códon: GCTGTAGTAAG
Terceira base de cada códon: AGGCATGCATC
Terceira base de cada códon: GCTGTAGTAAG
Sequência invertida: CGCTAAAAGCTAGGATATATCCGGGTAGCTAG


In [11]:
''' Transformando Seq em String '''

print('Objeto Seq:', my_seq_1)

# Formato FASTA simples (Lembrar para depois)
fasta_format_string = f'>Name\n{my_seq_1}\n'
print(fasta_format_string)

Objeto Seq: GATCG
>Name
GATCG



In [12]:
''' Concatenar ou adicionar sequências '''

seq1 = Seq('ACGT')
seq2 = Seq('AACCGG')
simple_concat = seq1 + seq2
print('Concatenação simples:', simple_concat)

# mal uso:
protein_seq = Seq('EVRNAK')
dna_seq = Seq('ACGT')
bad_concat = protein_seq + dna_seq
print('Misturando proteína com DNA (biologicamente errado):', bad_concat)

# listas de sequências (join)
list_of_seqs = [Seq('ACGT'), Seq('AACC'), Seq('GGTT')]

# Concatenação usando +=
concatenated_1 = Seq('')
for s in list_of_seqs:
    concatenated_1 += s
print('Concatenando usando +=:\t\t', concatenated_1)

# Concatenação usando join
concatenated_2 = Seq('').join(list_of_seqs)

print('Concatenando usando join:\t', concatenated_2)

Concatenação simples: ACGTAACCGG
Misturando proteína com DNA (biologicamente errado): EVRNAKACGT
Concatenando usando +=:		 ACGTAACCGGTT
Concatenando usando join:	 ACGTAACCGGTT


In [13]:
''' capslock '''

dna_seq_1 = Seq('acgtACGT')
print(dna_seq_1.upper())
print(dna_seq_1.lower())

ACGTACGT
acgtacgt


Uso de Seq com funções biopython

DNA $\to$ RNA $\to$ Proteína


#### Métodos do Seq


Aqui veremos alguns dos métodos mais comuns do objeto `Seq` para manipular sequências biológicas. Esses métodos permitem realizar operações como transcrição, tradução, complementação, entre outras, facilitando a análise e manipulação de sequências de DNA, RNA e proteínas.


In [14]:
''' Cálculo de GC% manual '''

gc_seq = Seq('GATCGATGGGCCTATATAGGATCGAAAATCGC')
print('Tamanho da sequência:', len(gc_seq))
print('Número de G:', gc_seq.count('G'))
gc_manual = 100 * (gc_seq.count('G') + gc_seq.count('C')) / len(gc_seq)
print('GC% manual:', gc_manual)

Tamanho da sequência: 32
Número de G: 9
GC% manual: 46.875


In [15]:
from Bio.SeqUtils import gc_fraction

''' Cálculo de GC% com função do Biopython '''

print(f'GC% usando função do Biopython: {100 * gc_fraction(gc_seq)}')

GC% usando função do Biopython: 46.875


In [16]:
''' Complemento, reverso complementar, transcrição e tradução '''

dna = Seq('ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG')

print('sequência original:', dna)
print('fita complementar:', dna.complement())
print('reverso complementar:', dna.reverse_complement())
print('DNA -> RNA:', dna.transcribe())
print('DNA -> proteína:', dna.translate())

sequência original: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
fita complementar: TACCGGTAACATTACCCGGCGACTTTCCCACGGGCTATC
reverso complementar: CTATCGGGCACCCTTTCAGCGGCCCATTACAATGGCCAT
DNA -> RNA: AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAG
DNA -> proteína: MAIVMGR*KGAR*


In [17]:
''' Fita codificadora e Fita Molde '''

coding_dna = Seq('ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG')
print('Fita codificadora (DNA):\t', coding_dna)

# A fita molde é o complemento reverso
template_dna = coding_dna.reverse_complement()
print('Fita molde (DNA):\t\t', template_dna)

Fita codificadora (DNA):	 ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
Fita molde (DNA):		 CTATCGGGCACCCTTTCAGCGGCCCATTACAATGGCCAT


In [18]:
''' Transcrição DNA -> RNA '''

# Transcrição "bioinformática' (da fita codificadora)
# Apenas substitui T por U
messenger_rna = coding_dna.transcribe()
print('mRNA transcrito da fita codificadora:\t', messenger_rna)

# Transcrição biológica real
real_mrna = messenger_rna.back_transcribe()
print('mRNA a partir da fita molde:\t\t', real_mrna)

mRNA transcrito da fita codificadora:	 AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAG
mRNA a partir da fita molde:		 ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG


In [19]:
''' Tradução '''

print('Tradução do mRNA:')
protein_from_rna = messenger_rna.translate()
print(protein_from_rna)

print('Tradução direto do DNA codificador:')
protein_from_dna = coding_dna.translate()
print(protein_from_dna)

Tradução do mRNA:
MAIVMGR*KGAR*
Tradução direto do DNA codificador:
MAIVMGR*KGAR*

MAIVMGR*KGAR*
Tradução direto do DNA codificador:
MAIVMGR*KGAR*


In [20]:
from Bio.Data import CodonTable

''' Tabela de tradução '''

standard_table = CodonTable.unambiguous_dna_by_name['Standard']
print(standard_table)

Table 1 Standard, SGC0

  |  T      |  C      |  A      |  G      |
--+---------+---------+---------+---------+--
T | TTT F   | TCT S   | TAT Y   | TGT C   | T
T | TTC F   | TCC S   | TAC Y   | TGC C   | C
T | TTA L   | TCA S   | TAA Stop| TGA Stop| A
T | TTG L(s)| TCG S   | TAG Stop| TGG W   | G
--+---------+---------+---------+---------+--
C | CTT L   | CCT P   | CAT H   | CGT R   | T
C | CTC L   | CCC P   | CAC H   | CGC R   | C
C | CTA L   | CCA P   | CAA Q   | CGA R   | A
C | CTG L(s)| CCG P   | CAG Q   | CGG R   | G
--+---------+---------+---------+---------+--
A | ATT I   | ACT T   | AAT N   | AGT S   | T
A | ATC I   | ACC T   | AAC N   | AGC S   | C
A | ATA I   | ACA T   | AAA K   | AGA R   | A
A | ATG M(s)| ACG T   | AAG K   | AGG R   | G
--+---------+---------+---------+---------+--
G | GTT V   | GCT A   | GAT D   | GGT G   | T
G | GTC V   | GCC A   | GAC D   | GGC G   | C
G | GTA V   | GCA A   | GAA E   | GGA G   | A
G | GTG V   | GCG A   | GAG E   | GGG G   | G
--+---------

In [21]:
''' Paradas biológicas '''

print('Tradução padrão:')
print(coding_dna.translate(), '\n')

print('Tradução até o primeiro códon STOP:')
print(coding_dna.translate(to_stop=True), '\n')

# Alterando símbolo de parada
print('Símbolo de parada personalizado:')
print(coding_dna.translate(table=2, stop_symbol='@'))

Tradução padrão:
MAIVMGR*KGAR* 

Tradução até o primeiro códon STOP:
MAIVMGR 

Símbolo de parada personalizado:
MAIVMGRWKGAR@


In [22]:
''' Tradução de CDS '''

# CDS = começa com códon start válido, termina com stop,
# não tem stops internos e comprimento múltiplo de 3

gene = Seq(
    'GTGAAAAAGATGCAATCTATCGTACTCGCACTTTCCCTGGTTCTGGTCGCTCCCATGGCA'
    'GCACAGGCTGCGGAAATTACGTTAGTCCCGTCAGTAAAATTACAGATAGGCGATCGTGAT'
    'AATCGTGGCTATTACTGGGATGGAGGTCACTGGCGCGACCACGGCTGGTGGAAACAACAT'
    'TATGAATGGCGAGGCAATCGCTGGCACCTACACGGACCGCCGCCACCGCCGCGCCACCAT'
    'AAGAAAGCTCCTCATGATCATCACGGCGGTCATGGTCCAGGCAAACATCACCGCTAA'
)

print('Tradução bacteriana normal:')
print(gene.translate(table='Bacterial')[:60], '...\n')

print('Tradução bacteriana até STOP:')
print(gene.translate(table='Bacterial', to_stop=True)[:60], '...\n')

# Aqui o GTG é códon start alternativo -> vira Metionina (M)
print('Tradução como CDS completa (start alternativo tratado):')
print(gene.translate(table='Bacterial', cds=True)[:60], '...')

Tradução bacteriana normal:
VKKMQSIVLALSLVLVAPMAAQAAEITLVPSVKLQIGDRDNRGYYWDGGHWRDHGWWKQH ...

Tradução bacteriana até STOP:
VKKMQSIVLALSLVLVAPMAAQAAEITLVPSVKLQIGDRDNRGYYWDGGHWRDHGWWKQH ...

Tradução como CDS completa (start alternativo tratado):
MKKMQSIVLALSLVLVAPMAAQAAEITLVPSVKLQIGDRDNRGYYWDGGHWRDHGWWKQH ...


### Tratamento de erros (Try ... Except)


In [23]:
''' Tratamento de erros com try-except-else '''

small_seq = Seq('ATG')

# Execução correta:
print(f'Índice 0: {small_seq[0]}')  # Índice 0: 'A'
print(f'Índice 1: {small_seq[1]}')  # Índice 1: 'T'
print(f'Índice 2: {small_seq[2]}')  # Índice 2: 'C'

# Execução com erro:

# print(f'Índice 3: {small_seq[3]}')

# Erro: "`IndexError`: list index out of range"

Índice 0: A
Índice 1: T
Índice 2: G


Se executássemos o código `print(f'Índice 3: {small_seq[3]}')` obteríamos o erro "`IndexError`: list index out of range", isso porque a sequência não tem uma quarta base nitrogenada. Então, para não gerarmos uma falha crítica na execução do código, podemos utilizar do **tratamento de erros**.

Uma forma trivial de tratar erros seria checar com um `if` se o índice existe ou não...


In [24]:
''' Tratamento de erros com if-else '''

if len(small_seq) > 3:  # Primeiro verifica se o tamanho contempla aquele índice
    # Existindo, ele mostra o valor naquele índice
    print(f'Índice 3: {small_seq[3]}')
else:  # Caso não exista...
    print('Esse índice não existe')  # Ele retorna aqui

Esse índice não existe


Mas isso pode ser trabalhoso e não é a melhor prática. O Python oferece uma estrutura de controle chamada `try ... except` que permite lidar com exceções de forma mais elegante e eficiente. Com essa estrutura, podemos tentar executar um bloco de código e, caso ocorra um erro, capturá-lo e tratá-lo sem interromper a execução do programa.

> Mas e se eu sei que algum erro pode ocorrer mas não sei qual?

Idealmente devemos nos antever para os erros específicos que podem ocorrer e que esperamos para que possamos lidar apropriadamente com cada um deles. Porém, como muita coisa inesperada pode acontecer, podemos tratá-los de forma mais generalista da seguinte forma:


In [25]:
''' Tratamento de erros com try-except '''

try:  # Tente...
    print(small_seq[3])  # Acessar o índice 3
# Caso ocorra alguma exceção que chamaremos de "e" por simplicidade...
except Exception as e:
    print('Ocorreu o erro:', e)

Ocorreu o erro: index out of range


> Mas e pra outros erros? Ou erros específicos?

Como vimos anteriormente, já temos ciência que o erro gerado é o `IndexError: index out of range`. Então, para lidar especificamente com esse erro, podemos prever esse erro em específico. E lidar com ele de forma apropriada, como por exemplo, informando ao usuário que o índice não existe.


In [26]:
''' Tratamento de erros sem especificar o tipo de erro '''

try:
    print(small_seq[3])
except IndexError:
    print('O índice em questão não existe, tente acessar um índice menor que o tamanho da sequência')

O índice em questão não existe, tente acessar um índice menor que o tamanho da sequência


Essa situação abaixo exemplifica o caso de uma sequência que existe como região anotada no genoma, mas que o conteúdo real ainda não foi sequenciado.


In [27]:
''' Sequência com conteúdo desconhecido; sabemos o comprimento, mas NÃO sabemos as letras '''

unknown_seq = Seq(None, 10)

print('Sequência desconhecida criada (conteúdo indefinido)')
print('Comprimento conhecido:', len(unknown_seq))
print('Representação interna:', repr(unknown_seq))

Sequência desconhecida criada (conteúdo indefinido)
Comprimento conhecido: 10
Representação interna: Seq(None, length=10)


Tentar acessar o conteúdo de uma sequência vazia/nula gera erro...


In [28]:
# print(unknown_seq[0]) # 'UndefinedSequenceError: Sequence content is undefined'

Isso ocorre porque, a sequência criada é indefinida, logo, não há nenhuma "base nitrogenada"/"caractere" na primeira posição. Assim causando o erro "UndefinedSequenceError"

Para evitar que o código quebre e pare como ocorreu na célula anterior, podemos usar o `try ... except` para contornarmos esse erro esperado.


In [29]:
from Bio.Seq import UndefinedSequenceError  # Importando o erro esperado

''' Tratamento de erros com try-except '''

try:  # Tente...
    # Imprimir a base nitrogenada na posição 0 da sequência desconhecida
    print(unknown_seq[0])
# Caso ocorra o erro "UndefinedSequenceError" que chamaremos de "e" por simplicidade...
except UndefinedSequenceError:  # Capturamos o erro específico, e...
    print('Erro ao acessar o conteúdo, lembre-se de que a sequência é desconhecida e não tem conteúdo definido')
except Exception as e:  # Captura qualquer outro erro que possa ocorrer, nomeando-o como "e"
    # Mostre o conteúdo do erro ocorrido
    print('Ocorreu um erro inesperado:', e)
else:  # Caso não ocorra nenhum erro, execute o código abaixo
    print('Não houve erros ao acessar o conteúdo da sequência desconhecida')

Erro ao acessar o conteúdo, lembre-se de que a sequência é desconhecida e não tem conteúdo definido


### Seq Avançado


In [30]:
''' Concatenando com regiões desconhecidas '''

seq_defined = Seq('ACGT')
undefined_seq = Seq(None, length=10)

combined = seq_defined + undefined_seq + seq_defined

# print("Concatenação com região indefinida:", combined) # Gera erro por indefinição

print('A sequência está definida?', combined.defined)
print('Intervalos definidos na sequência:', combined.defined_ranges)

# Criando uma string "segura" para imprimir, onde as regiões indefinidas são representadas por 'N'
safe_string = str(seq_defined) + 'N' * len(undefined_seq) + str(seq_defined)

print('Sequência segura para imprimir:', safe_string)

A sequência está definida? False
Intervalos definidos na sequência: ((0, 4), (14, 18))
Sequência segura para imprimir: ACGTNNNNNNNNNNACGT


In [31]:
''' Sequências parcialmente definidas; sabemos o comprimento, e algumas letras, mas não todas '''

# Sequência parcialmente definidas
partial_seq = Seq(
    {117512683: 'TTGAAAACCTGAATGTGAGAGTCAGTCAAGGATAGT'},
    length=159345973
)

print('Sequência parcialmente definida criada!')

# Região totalmente desconhecida
unknown_slice = partial_seq[1000:1020]
print('Trecho desconhecido (estrutura):', repr(unknown_slice))

# Região totalmente conhecida
known_slice = partial_seq[117512690:117512700]
print('Trecho conhecido:', known_slice)  # aqui pode imprimir normal

# Região parcialmente conhecida
partial_slice = partial_seq[117512670:117512690]
print('Trecho parcialmente conhecido (estrutura):', repr(partial_slice))

# Do meio até o final
tail_slice = partial_seq[117512700:]
print('Do meio até o fim (estrutura):', repr(tail_slice))

Sequência parcialmente definida criada!
Trecho desconhecido (estrutura): Seq(None, length=20)
Trecho conhecido: CCTGAATGTG
Trecho parcialmente conhecido (estrutura): Seq({13: 'TTGAAAA'}, length=20)
Do meio até o fim (estrutura): Seq({0: 'AGAGTCAGTCAAGGATAGT'}, length=41833273)


#### Objetos MutableSeq (sequências editáveis)


In [32]:
from Bio.Seq import MutableSeq

''' Sequências mutáveis '''

immutable_seq = Seq('GCCATTGTAATGGGCCGCTGAAAGGGTGCCCGA')

# Seq é imutável
try:
    immutable_seq[5] = 'G'
except TypeError as e:
    print('Seq é imutável:', e)

# Criando sequência mutável
mutable_seq = MutableSeq(immutable_seq)
print('MutableSeq inicial:\t\t', mutable_seq)

# Alterando base (mutação pontual)
mutable_seq[5] = 'C'
print('Após mutação:\t\t\t', mutable_seq)

# Removendo primeira ocorrência de uma base
mutable_seq.remove('T')

# isso não remove por posição, remove por valor
print('Após remover T:\t\t\t', mutable_seq)
del mutable_seq[3]

# isso removeu a terceira posição
print('Removendo a terceira base:\t', mutable_seq)

# Invertendo IN PLACE (modifica o próprio objeto)
mutable_seq.reverse()
print('Após reverse():\t\t\t', mutable_seq)

# Voltando para Seq imutável
new_seq = Seq(mutable_seq)

# salvou o resultado final (após reverse)
print('Convertido de volta para Seq:\t', new_seq)

Seq é imutável: 'Seq' object does not support item assignment
MutableSeq inicial:		 GCCATTGTAATGGGCCGCTGAAAGGGTGCCCGA
Após mutação:			 GCCATCGTAATGGGCCGCTGAAAGGGTGCCCGA
Após remover T:			 GCCACGTAATGGGCCGCTGAAAGGGTGCCCGA
Removendo a terceira base:	 GCCCGTAATGGGCCGCTGAAAGGGTGCCCGA
Após reverse():			 AGCCCGTGGGAAAGTCGCCGGGTAATGCCCG
Convertido de volta para Seq:	 AGCCCGTGGGAAAGTCGCCGGGTAATGCCCG


In [33]:
''' Busca de subsequências '''

seq = Seq('GCCATTGTAATGGGCCGCTGAAAGGGTGCCCGA')
subseq = 'ATGGGCCGC'

print('Busca usando diferentes tipos:')
print('\t- String:', seq.index(subseq))
print('\t- Lendo str como bytes:', seq.index(b'ATGGGCCGC'))
print('\t- Convertendo str para bytes:', seq.index(bytes(subseq, 'utf-8')))
print('\t- Usando bytearray:', seq.index(bytearray(b'ATGGGCCGC')))
print('\t- Buscando usando Seq:', seq.index(Seq(subseq)))
print('\t- Buscando usando MutableSeq:', seq.index(MutableSeq(subseq)))

# Diferença entre index e find
subseq = 'ACTG'
try:
    print(seq.index(subseq))
except ValueError:
    print('index() gera erro se não encontrar')

print('find() retorna -1 se não encontrar:', seq.find(subseq))

# Busca da esquerda vs direita
subseq = 'CC'
print(f'Primeira {subseq}:', seq.find(subseq))
print(f'Última {subseq}:', seq.rfind(subseq))  # última ocorrência

Busca usando diferentes tipos:
	- String: 9
	- Lendo str como bytes: 9
	- Convertendo str para bytes: 9
	- Usando bytearray: 9
	- Buscando usando Seq: 9
	- Buscando usando MutableSeq: 9
index() gera erro se não encontrar
find() retorna -1 se não encontrar: -1
Primeira CC: 1
Última CC: 29


In [34]:
''' Busca múltipla com search() '''

print('Sequência para busca múltipla:', seq)
print('Busca múltipla de padrões:')
for index, sub in seq.search(['CC', Seq('GGG'), 'CC']):
    print(index, '\t', sub)

Sequência para busca múltipla: GCCATTGTAATGGGCCGCTGAAAGGGTGCCCGA
Busca múltipla de padrões:
1 	 CC
11 	 GGG
14 	 CC
23 	 GGG
28 	 CC
29 	 CC


### Leitura e Escrita de Sequências: Seq IO


- Interfaces de leitura e escrita de arquivos de sequência em [diversos formatos (FASTA, GenBank, etc.)](https://biopython.org/docs/latest/api/Bio.SeqIO.html#file-formats):
  - `Bio.SeqIO`: interação padronizada para diferentes formatos de sequência
    - Preferível pois é mais simples e trabalha de forma padrão com os objetos SeqRecord
  - `Bio.AlignIO`: interação específica para cada formato de alinhamento
    - É mais flexível, mas requer mais conhecimento sobre os formatos de alinhamento

> **Formatos:** abi, abi-trim, ace, cif-atom, cif-seqres, clustal, embl, fasta, fasta-2line, fastq-sanger or fastq, fastq-solexa, fastq-illumina, gck, genbank or gb, ig, imgt, nexus, pdb-seqres, pdb-atom, phd, phylip, pir, seqxml, sff, sff-trim, snapgene, stockholm, swiss, tab, qual, uniprot-xml, xdna.

---

IO = Input/Output (Entrada/Saída)


#### Lendo arquivos


- **Leitura de arquivos de sequência: `Bio.SeqIO.Parse`**

Aqui indicamos o nome do arquivo e o formato, e o Biopython irá ler o arquivo e criar um objeto SeqRecord para cada sequência presente no arquivo. O método `parse` é um gerador, o que significa que ele retorna um iterador que pode ser usado para percorrer os registros um por um, sem carregar todo o arquivo na memória de uma vez.

São apresentadas duas formas de usar o `parse`:

1. Diretamente, passando o nome do arquivo e o formato.
2. Abrindo o arquivo manualmente e passando o objeto de arquivo para o `parse`.


In [35]:
from Bio import SeqIO

In [36]:
''' Demonstrando leitura de sequências '''

file_name = 'NC_045512.2.fasta'
file_format = 'fasta'

print('Records de...')

print('1. Parsing direto do arquivo:')

for seq_record in SeqIO.parse(file_name, file_format):
    print(f'\t- ID: {seq_record.id}')

# Alternativamente:

print('2. file opening')
with open(file_name) as fasta_file:
    for record in SeqIO.parse(fasta_file, file_format):
        print(f'\t- ID: {record.id}')

print('3. first record parsing')
fasta_record = SeqIO.read(file_name, file_format)
print(f'\t- ID: {fasta_record.id}')

Records de...
1. Parsing direto do arquivo:
	- ID: NC_045512.2
2. file opening
	- ID: NC_045512.2
3. first record parsing
	- ID: NC_045512.2


##### Lendo arquivo FASTA


Em seguida, mostramos como usar o `read` para ler um único registro de um arquivo FASTA. O `read` é usado quando se espera que o arquivo contenha apenas um registro, e ele retorna esse registro como um objeto SeqRecord. Se o arquivo contiver mais de um registro, o `read` lançará uma exceção.


In [37]:
''' Lendo arquivos FASTA com SeqIO '''

# exige exatamente 1 registro, ou seja, arquivo com uma única sequência
record = SeqIO.read('NC_005816.fna', 'fasta')

print(record.seq)  # retorna um objeto seq com a sequência inteira

# Identificadores
# print(record.id)
# print(record.name)
# print(record.description)

# Outras formas de obter o arquivo

# - Caminho completo do arquivo
# record = SeqIO.read('C:\\Users\\Nome\\Downloads\\NC_005816.fna', 'fasta')

TGTAACGAACGGTGCAATAGTGATCCACACCCAACGCCTGAAATCAGATCCAGGGGGTAATCTGCTCTCCTGATTCAGGAGAGTTTATGGTCACTTTTGAGACAGTTATGGAAATTAAAATCCTGCACAAGCAGGGAATGAGTAGCCGGGCGATTGCCAGAGAACTGGGGATCTCCCGCAATACCGTTAAACGTTATTTGCAGGCAAAATCTGAGCCGCCAAAATATACGCCGCGACCTGCTGTTGCTTCACTCCTGGATGAATACCGGGATTATATTCGTCAACGCATCGCCGATGCTCATCCTTACAAAATCCCGGCAACGGTAATCGCTCGCGAGATCAGAGACCAGGGATATCGTGGCGGAATGACCATTCTCAGGGCATTCATTCGTTCTCTCTCGGTTCCTCAGGAGCAGGAGCCTGCCGTTCGGTTCGAAACTGAACCCGGACGACAGATGCAGGTTGACTGGGGCACTATGCGTAATGGTCGCTCACCGCTTCACGTGTTCGTTGCTGTTCTCGGATACAGCCGAATGCTGTACATCGAATTCACTGACAATATGCGTTATGACACGCTGGAGACCTGCCATCGTAATGCGTTCCGCTTCTTTGGTGGTGTGCCGCGCGAAGTGTTGTATGACAATATGAAAACTGTGGTTCTGCAACGTGACGCATATCAGACCGGTCAGCACCGGTTCCATCCTTCGCTGTGGCAGTTCGGCAAGGAGATGGGCTTCTCTCCCCGACTGTGTCGCCCCTTCAGGGCACAGACTAAAGGTAAGGTGGAACGGATGGTGCAGTACACCCGTAACAGTTTTTACATCCCACTAATGACTCGCCTGCGCCCGATGGGGATCACTGTCGATGTTGAAACAGCCAACCGCCACGGTCTGCGCTGGCTGCACGATGTCGCTAACCAACGAAAGCATGAAACAATCCAGGCCCGTCCCTGCGATCGCTGGCTCGAAGAGCAGCAGTCCATGCTGGCACTGCCTCCGGA

In [38]:
''' Extraindo Dados do FASTA '''

all_species = []
for seq_record in SeqIO.parse('ls_orchid.fasta', 'fasta'):
    all_species.append(seq_record.description.split()[1])

print(all_species)

['C.irapeanum', 'C.californicum', 'C.fasciculatum', 'C.margaritaceum', 'C.lichiangense', 'C.yatabeanum', 'C.guttatum', 'C.acaule', 'C.formosanum', 'C.himalaicum', 'C.macranthum', 'C.calceolus', 'C.segawai', 'C.pubescens', 'C.reginae', 'C.flavum', 'C.passerinum', 'M.xerophyticum', 'P.schlimii', 'P.besseae', 'P.wallisii', 'P.exstaminodium', 'P.caricinum', 'P.pearcei', 'P.longifolium', 'P.lindenii', 'P.lindleyanum', 'P.sargentianum', 'P.kaiteurum', 'P.czerwiakowianum', 'P.boissierianum', 'P.caudatum', 'P.warszewiczianum', 'P.micranthum', 'P.malipoense', 'P.delenatii', 'P.armeniacum', 'P.emersonii', 'P.niveum', 'P.godefroyae', 'P.bellatulum', 'P.concolor', 'P.fairrieanum', 'P.druryi', 'P.tigrinum', 'P.hirsutissimum', 'P.barbigerum', 'P.henryanum', 'P.charlesworthii', 'P.villosum', 'P.exul', 'P.insigne', 'P.gratrixianum', 'P.primulinum', 'P.victoria', 'P.victoria', 'P.glaucophyllum', 'P.supardii', 'P.kolopakingii', 'P.sanderianum', 'P.lowii', 'P.dianthum', 'P.parishii', 'P.haynaldianum', 'P

In [39]:
from Bio.SeqIO.FastaIO import SimpleFastaParser
# from Bio.SeqIO.QualityIO import FastqGeneralIterator

''' Parses (de baixo nível) para FASTA e FASTQ '''
# Esse parser lê FASTA e devolver strings puras, não objetos

count = 0
total_len = 0

# with open('example.fastq') as in_handle:
with open('ls_orchid.fasta') as in_handle:
    # for title, seq in FastqGeneralIterator(in_handle):
    for title, seq in SimpleFastaParser(in_handle):
        count += 1
        total_len += len(seq)

print(f'{count} records with total sequence length {total_len}')

94 records with total sequence length 67518


##### Lendo arquivo GenBank


In [40]:
''' Lendo arquivos GenBank com SeqIO '''

record = SeqIO.read('NC_005816.gb', 'genbank')

print(record.seq)  # a sequência também está aqui
print(record.annotations)
print(record.annotations['source'])
print(len(record.features))

for feature in record.features[:5]:
    print(feature.type)
    print(feature.location)
    print(feature.qualifiers)
    print('-----')

print(record.letter_annotations)
# mas essa parte continua vazia, pois GenBank normalmente não traz informações por base
# (ex: qualidade de sequenciamento, isso é típico de FASTQ)

TGTAACGAACGGTGCAATAGTGATCCACACCCAACGCCTGAAATCAGATCCAGGGGGTAATCTGCTCTCCTGATTCAGGAGAGTTTATGGTCACTTTTGAGACAGTTATGGAAATTAAAATCCTGCACAAGCAGGGAATGAGTAGCCGGGCGATTGCCAGAGAACTGGGGATCTCCCGCAATACCGTTAAACGTTATTTGCAGGCAAAATCTGAGCCGCCAAAATATACGCCGCGACCTGCTGTTGCTTCACTCCTGGATGAATACCGGGATTATATTCGTCAACGCATCGCCGATGCTCATCCTTACAAAATCCCGGCAACGGTAATCGCTCGCGAGATCAGAGACCAGGGATATCGTGGCGGAATGACCATTCTCAGGGCATTCATTCGTTCTCTCTCGGTTCCTCAGGAGCAGGAGCCTGCCGTTCGGTTCGAAACTGAACCCGGACGACAGATGCAGGTTGACTGGGGCACTATGCGTAATGGTCGCTCACCGCTTCACGTGTTCGTTGCTGTTCTCGGATACAGCCGAATGCTGTACATCGAATTCACTGACAATATGCGTTATGACACGCTGGAGACCTGCCATCGTAATGCGTTCCGCTTCTTTGGTGGTGTGCCGCGCGAAGTGTTGTATGACAATATGAAAACTGTGGTTCTGCAACGTGACGCATATCAGACCGGTCAGCACCGGTTCCATCCTTCGCTGTGGCAGTTCGGCAAGGAGATGGGCTTCTCTCCCCGACTGTGTCGCCCCTTCAGGGCACAGACTAAAGGTAAGGTGGAACGGATGGTGCAGTACACCCGTAACAGTTTTTACATCCCACTAATGACTCGCCTGCGCCCGATGGGGATCACTGTCGATGTTGAAACAGCCAACCGCCACGGTCTGCGCTGGCTGCACGATGTCGCTAACCAACGAAAGCATGAAACAATCCAGGCCCGTCCCTGCGATCGCTGGCTCGAAGAGCAGCAGTCCATGCTGGCACTGCCTCCGGA

##### Lendo arquivos compactados (gzip)


In [41]:
import gzip  # Ou import bz2 para arquivos .bz2

''' Analisando sequências de arquivos compactados '''

file_name = 'ls_orchid.gbk.gz'
file_format = 'gb'

# o mesmo serve para .bz2, só mudar 'gzip' para 'bz2':
# with modulo_de_compactacao.open(file_name, 'rt') as handle:
with gzip.open(file_name, 'rt') as handle:
    for record in SeqIO.parse(handle, file_format):
        print(f'ID: {record.id}, Tamanho: {len(record)}')
    print(sum(len(r) for r in SeqIO.parse(handle, 'gb')))

ID: Z78533.1, Tamanho: 740
ID: Z78532.1, Tamanho: 753
ID: Z78531.1, Tamanho: 748
ID: Z78530.1, Tamanho: 744
ID: Z78529.1, Tamanho: 733
ID: Z78527.1, Tamanho: 718
ID: Z78526.1, Tamanho: 730
ID: Z78525.1, Tamanho: 704
ID: Z78524.1, Tamanho: 740
ID: Z78523.1, Tamanho: 709
ID: Z78522.1, Tamanho: 700
ID: Z78521.1, Tamanho: 726
ID: Z78520.1, Tamanho: 753
ID: Z78519.1, Tamanho: 699
ID: Z78518.1, Tamanho: 658
ID: Z78517.1, Tamanho: 752
ID: Z78516.1, Tamanho: 726
ID: Z78515.1, Tamanho: 765
ID: Z78514.1, Tamanho: 755
ID: Z78513.1, Tamanho: 742
ID: Z78512.1, Tamanho: 762
ID: Z78511.1, Tamanho: 745
ID: Z78510.1, Tamanho: 750
ID: Z78509.1, Tamanho: 731
ID: Z78508.1, Tamanho: 741
ID: Z78507.1, Tamanho: 740
ID: Z78506.1, Tamanho: 727
ID: Z78505.1, Tamanho: 711
ID: Z78504.1, Tamanho: 743
ID: Z78503.1, Tamanho: 727
ID: Z78502.1, Tamanho: 757
ID: Z78501.1, Tamanho: 770
ID: Z78500.1, Tamanho: 767
ID: Z78499.1, Tamanho: 759
ID: Z78498.1, Tamanho: 750
ID: Z78497.1, Tamanho: 788
ID: Z78496.1, Tamanho: 774
I

In [42]:
''' Lendo arquivos 2bit '''

# Em arquivos 2bit já são indexados internamente, ou seja, o SeqIO.parse() já se comporta como um dicionário

# open('sequence_example.bigendian.2bit', 'rb') as handle:
# records = SeqIO.parse(handle, 'twobit')
# records.keys()

# como a leitura é sob demanda, o arquivo precisa ficar aberto, então:
# handle.close()

' Lendo arquivos 2bit '

##### Carregando o arquivo para a memória


Para registros pequenos, a otimização de memória do `parse` não é tão necessária, portanto, para facilitar o acesso aos dados, podemos convertê-lo em uma lista usando a função `list()`. Isso carregará todos os registros na memória, permitindo que você acesse cada um deles diretamente por meio de índices. No entanto, é importante ter cuidado ao usar essa abordagem com arquivos grandes, pois pode consumir muita memória. Outra alternativa é usar o `SeqIO.to_dict()`, que permite criar um dicionário de registros, onde as chaves são os IDs dos registros e os valores são os objetos SeqRecord correspondentes. Isso pode ser útil para acessar rapidamente um registro específico através de seu ID.


In [43]:
'''  Lendo o arquivo inteiro para a memória '''

file_name = 'ls_orchid.gbk'
file_format = 'genbank'

# Permite ler o arquivo inteiro para a memória, criando uma lista de SeqRecords
lista_de_records = list(SeqIO.parse(file_name, file_format))
# Com isso, podemos acessar arbitrariamente a sequência (fora de ordem, saber quantas existem, etc)

# Descobrindo quantas sequências existem
print(f'Há {len(lista_de_records)} sequências no arquivo.')

# Pegando o último registro
print(f'Último ID: {lista_de_records[-1].id}')

# Também é possível criar um dicionário, onde as chaves são os IDs dos records, e os valores são os próprios SeqRecords
orchid_dict = SeqIO.to_dict(SeqIO.parse(file_name, file_format))
# Esse é um banco em memória (alta), tem alta velocidade e pode editar
print(len(orchid_dict))
print(list(orchid_dict.keys()))

# Lista de ID:
seq_record = orchid_dict['Z78475.1']
print(seq_record.description)
print(seq_record.seq)
# Vantagem: acesso instantâneo, mas carrega tudo na RAM

orchid_dict_idx = SeqIO.index(file_name, file_format)
# Índice no arquivo, não carrega tudo na memória, lê só o que precisa
# Não pode modificar os registros, mas gasta menos memória

orchid_dict_db = SeqIO.index_db('orchid.idx', file_name, file_format)
# Aqui o índice é salvo em disco, quase não usa RAM, porém é mais lento e funciona somente leitura

Há 94 sequências no arquivo.
Último ID: Z78439.1
94
['Z78533.1', 'Z78532.1', 'Z78531.1', 'Z78530.1', 'Z78529.1', 'Z78527.1', 'Z78526.1', 'Z78525.1', 'Z78524.1', 'Z78523.1', 'Z78522.1', 'Z78521.1', 'Z78520.1', 'Z78519.1', 'Z78518.1', 'Z78517.1', 'Z78516.1', 'Z78515.1', 'Z78514.1', 'Z78513.1', 'Z78512.1', 'Z78511.1', 'Z78510.1', 'Z78509.1', 'Z78508.1', 'Z78507.1', 'Z78506.1', 'Z78505.1', 'Z78504.1', 'Z78503.1', 'Z78502.1', 'Z78501.1', 'Z78500.1', 'Z78499.1', 'Z78498.1', 'Z78497.1', 'Z78496.1', 'Z78495.1', 'Z78494.1', 'Z78493.1', 'Z78492.1', 'Z78491.1', 'Z78490.1', 'Z78489.1', 'Z78488.1', 'Z78487.1', 'Z78486.1', 'Z78485.1', 'Z78484.1', 'Z78483.1', 'Z78482.1', 'Z78481.1', 'Z78480.1', 'Z78479.1', 'Z78478.1', 'Z78477.1', 'Z78476.1', 'Z78475.1', 'Z78474.1', 'Z78473.1', 'Z78472.1', 'Z78471.1', 'Z78470.1', 'Z78469.1', 'Z78468.1', 'Z78467.1', 'Z78466.1', 'Z78465.1', 'Z78464.1', 'Z78463.1', 'Z78462.1', 'Z78461.1', 'Z78460.1', 'Z78459.1', 'Z78458.1', 'Z78457.1', 'Z78456.1', 'Z78455.1', 'Z78454.1',

- SeqIO.to_dict vs SeqIO.index:
  - to_dict: carrega tudo na memória
  - index: carrega sob demanda

Para sequências muito grandes, sugere-se usar o BioSQL.


##### Percorrendo registros


De um modo geral, O `for ... in range()` é usado quando o número de iterações é conhecido antes do início do loop. O `next` é usado para pular para o próximo item dentro de um iterador (um iterador é um objeto python que é eficiente em processamento de dados, não sendo necessário armazenar todos os elementos na memória). E o `while` é usado para percorrer um iterador até que uma condição seja satisfeita, como por exemplo, até que não haja mais itens para iterar.


In [44]:
''' Demonstrando processamento de sequências em um fasta '''

for seq_record in SeqIO.parse('ls_orchid.fasta', 'fasta'):
    # Demonstrando prints sequenciais
    print('Nome da sequência:', seq_record.id, end='; ')
    print('Sequência:', repr(seq_record.seq), end='; ')
    print('Tamanho:', len(seq_record))

Nome da sequência: gi|2765658|emb|Z78533.1|CIZ78533; Sequência: Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGATGAGACCGTGG...CGC'); Tamanho: 740
Nome da sequência: gi|2765657|emb|Z78532.1|CCZ78532; Sequência: Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGTTGAGACAACAG...GGC'); Tamanho: 753
Nome da sequência: gi|2765656|emb|Z78531.1|CFZ78531; Sequência: Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGTTGAGACAGCAG...TAA'); Tamanho: 748
Nome da sequência: gi|2765655|emb|Z78530.1|CMZ78530; Sequência: Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGTTGAAACAACAT...CAT'); Tamanho: 744
Nome da sequência: gi|2765654|emb|Z78529.1|CLZ78529; Sequência: Seq('ACGGCGAGCTGCCGAAGGACATTGTTGAGACAGCAGAATATACGATTGAGTGAA...AAA'); Tamanho: 733
Nome da sequência: gi|2765652|emb|Z78527.1|CYZ78527; Sequência: Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGTTGAGACAGTAG...CCC'); Tamanho: 718
Nome da sequência: gi|2765651|emb|Z78526.1|CGZ78526; Sequência: Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGTTGAGACAGTAG.

In [45]:
''' for ... in ... range '''

index = 0
for seq_record in SeqIO.parse('ls_orchid.fasta', 'fasta'):
    index_and_size = f'Índice: {index}; Tamanho: {len(seq_record)}'
    print(index_and_size)  # Demonstrando o uso do índice
    index += 1

Índice: 0; Tamanho: 740
Índice: 1; Tamanho: 753
Índice: 2; Tamanho: 748
Índice: 3; Tamanho: 744
Índice: 4; Tamanho: 733
Índice: 5; Tamanho: 718
Índice: 6; Tamanho: 730
Índice: 7; Tamanho: 704
Índice: 8; Tamanho: 740
Índice: 9; Tamanho: 709
Índice: 10; Tamanho: 700
Índice: 11; Tamanho: 726
Índice: 12; Tamanho: 753
Índice: 13; Tamanho: 699
Índice: 14; Tamanho: 658
Índice: 15; Tamanho: 752
Índice: 16; Tamanho: 726
Índice: 17; Tamanho: 765
Índice: 18; Tamanho: 755
Índice: 19; Tamanho: 742
Índice: 20; Tamanho: 762
Índice: 21; Tamanho: 745
Índice: 22; Tamanho: 750
Índice: 23; Tamanho: 731
Índice: 24; Tamanho: 741
Índice: 25; Tamanho: 740
Índice: 26; Tamanho: 727
Índice: 27; Tamanho: 711
Índice: 28; Tamanho: 743
Índice: 29; Tamanho: 727
Índice: 30; Tamanho: 757
Índice: 31; Tamanho: 770
Índice: 32; Tamanho: 767
Índice: 33; Tamanho: 759
Índice: 34; Tamanho: 750
Índice: 35; Tamanho: 788
Índice: 36; Tamanho: 774
Índice: 37; Tamanho: 789
Índice: 38; Tamanho: 688
Índice: 39; Tamanho: 719
Índice: 40

In [46]:
''' for ... in ... enumerate '''

for index, seq_record in enumerate(SeqIO.parse('ls_orchid.gbk', 'genbank')):
    # Demonstrando outro uso do índice
    print(f'Índice: {index}; Tamanho: {len(seq_record)}')

Índice: 0; Tamanho: 740
Índice: 1; Tamanho: 753
Índice: 2; Tamanho: 748
Índice: 3; Tamanho: 744
Índice: 4; Tamanho: 733
Índice: 5; Tamanho: 718
Índice: 6; Tamanho: 730
Índice: 7; Tamanho: 704
Índice: 8; Tamanho: 740
Índice: 9; Tamanho: 709
Índice: 10; Tamanho: 700
Índice: 11; Tamanho: 726
Índice: 12; Tamanho: 753
Índice: 13; Tamanho: 699
Índice: 14; Tamanho: 658
Índice: 15; Tamanho: 752
Índice: 16; Tamanho: 726
Índice: 17; Tamanho: 765
Índice: 18; Tamanho: 755
Índice: 19; Tamanho: 742
Índice: 20; Tamanho: 762
Índice: 21; Tamanho: 745
Índice: 22; Tamanho: 750
Índice: 23; Tamanho: 731
Índice: 24; Tamanho: 741
Índice: 25; Tamanho: 740
Índice: 26; Tamanho: 727
Índice: 27; Tamanho: 711
Índice: 28; Tamanho: 743
Índice: 29; Tamanho: 727
Índice: 30; Tamanho: 757
Índice: 31; Tamanho: 770
Índice: 32; Tamanho: 767
Índice: 33; Tamanho: 759
Índice: 34; Tamanho: 750
Índice: 35; Tamanho: 788
Índice: 36; Tamanho: 774
Índice: 37; Tamanho: 789
Índice: 38; Tamanho: 688
Índice: 39; Tamanho: 719
Índice: 40

In [47]:
''' Percorrendo um iterador com while e next '''

records = SeqIO.parse('ls_orchid.gbk', 'genbank')
index = 0
while True:
    try:
        seq_record = next(records)  # Pega o próximo record
        print(f'Índice: {index}; Tamanho: {len(seq_record)}')
        index += 1
    # Quando o iterador for esgotado (terminarmos de percorrer todas as sequências), ele retornará um erro, e nesse caso...
    except StopIteration:
        break  # Precisamos parar o loop

Índice: 0; Tamanho: 740
Índice: 1; Tamanho: 753
Índice: 2; Tamanho: 748
Índice: 3; Tamanho: 744
Índice: 4; Tamanho: 733
Índice: 5; Tamanho: 718
Índice: 6; Tamanho: 730
Índice: 7; Tamanho: 704
Índice: 8; Tamanho: 740
Índice: 9; Tamanho: 709
Índice: 10; Tamanho: 700
Índice: 11; Tamanho: 726
Índice: 12; Tamanho: 753
Índice: 13; Tamanho: 699
Índice: 14; Tamanho: 658
Índice: 15; Tamanho: 752
Índice: 16; Tamanho: 726
Índice: 17; Tamanho: 765
Índice: 18; Tamanho: 755
Índice: 19; Tamanho: 742
Índice: 20; Tamanho: 762
Índice: 21; Tamanho: 745
Índice: 22; Tamanho: 750
Índice: 23; Tamanho: 731
Índice: 24; Tamanho: 741
Índice: 25; Tamanho: 740
Índice: 26; Tamanho: 727
Índice: 27; Tamanho: 711
Índice: 28; Tamanho: 743
Índice: 29; Tamanho: 727
Índice: 30; Tamanho: 757
Índice: 31; Tamanho: 770
Índice: 32; Tamanho: 767
Índice: 33; Tamanho: 759
Índice: 34; Tamanho: 750
Índice: 35; Tamanho: 788
Índice: 36; Tamanho: 774
Índice: 37; Tamanho: 789
Índice: 38; Tamanho: 688
Índice: 39; Tamanho: 719
Índice: 40

In [48]:
''' Leitura sem o for, mas com .next() '''

# Acessar só o primeiro registro, bom para testar o arquivo
first_record = next(SeqIO.parse('ls_orchid.gbk', 'genbank'))
print(first_record)

# Extraindo os dados do GenBank

# Acessando as anotações
print(first_record.annotations)

# Pegando chaves e valores
print(first_record.annotations.keys())    # só as 'categorias'
print(first_record.annotations.values())  # só os valores

# Extraindo o nome da espécie
print(first_record.annotations['source'])
print(first_record.annotations['organism'])

# Pegando todas as espécies do arquivo GenBank
all_species = []
# o arquivo é percorrido registro por registro
for seq_record in SeqIO.parse('ls_orchid.gbk', 'genbank'):
    # para cada sequência: acessamos annotations['organism']; salva tudo numa lista
    all_species.append(seq_record.annotations['organism'])

print(all_species)

ID: Z78533.1
Name: Z78533
Description: C.irapeanum 5.8S rRNA gene and ITS1 and ITS2 DNA
Number of features: 5
/molecule_type=DNA
/topology=linear
/data_file_division=PLN
/date=30-NOV-2006
/accessions=['Z78533']
/sequence_version=1
/gi=2765658
/keywords=['5.8S ribosomal RNA', '5.8S rRNA gene', 'internal transcribed spacer', 'ITS1', 'ITS2']
/source=Cypripedium irapeanum
/organism=Cypripedium irapeanum
/taxonomy=['Eukaryota', 'Viridiplantae', 'Streptophyta', 'Embryophyta', 'Tracheophyta', 'Spermatophyta', 'Magnoliophyta', 'Liliopsida', 'Asparagales', 'Orchidaceae', 'Cypripedioideae', 'Cypripedium']
/references=[Reference(title='Phylogenetics of the slipper orchids (Cypripedioideae: Orchidaceae): nuclear rDNA ITS sequences', ...), Reference(title='Direct Submission', ...)]
Seq('CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGATGAGACCGTGG...CGC')
{'molecule_type': 'DNA', 'topology': 'linear', 'data_file_division': 'PLN', 'date': '30-NOV-2006', 'accessions': ['Z78533'], 'sequence_version': 1, 'gi'

#### Escrevendo/Convertendo arquivos


In [49]:
''' Conversão entre formatos de arquivos de sequência '''

records = SeqIO.parse('ls_orchid.gbk', 'genbank')  # Lê GenBank
count = SeqIO.write(records, 'my_example.fasta', 'fasta')  # Escreve FASTA

print(f'Converted {count} records')

# Função .convert()

count = SeqIO.convert('ls_orchid.gbk', 'genbank', 'my_example.fasta', 'fasta')

print(f'Converted {count} records')

# Isso funciona para FASTQ -> FASTA, mas não ao contrário

Converted 94 records
Converted 94 records


### Seq Record


**Seq** é só a sequência.

**SeqRecord** é a sequência com identidade, contexto biológico e anotações.


In [50]:
from Bio.SeqRecord import SeqRecord

''' SeqRecord: um registro de sequência com metadados '''

simple_seq = Seq('GATC')  # Previamente importado do módulo Bio.Seq
simple_seq_r = SeqRecord(simple_seq)  # registro vazio

# o biopython preenche com valores padrão quando você não fornece
print(simple_seq_r.id)

# Definindo identidade
simple_seq_r.id = 'AC12345'  # essencial para salvar em FASTA ou GenBank depois
simple_seq_r.description = 'Sequência Exemplo'

print(simple_seq_r.id)
print(simple_seq_r.description)

# Adicionando anotações gerais
simple_seq_r.annotations['evidence'] = 'Sequência simulada'
simple_seq_r.annotations['organism'] = 'Synthetic construct'
print(simple_seq_r.annotations)

# Anotação por base
simple_seq_r.letter_annotations['phred_quality'] = [40, 40, 38, 30]
# A lista PRECISA ter o mesmo tamanho da sequência, caso contrário vai dar erro

# Referências externas
simple_seq_r.dbxrefs.append('GeneID:0000')
simple_seq_r.dbxrefs.append('Project:DidacticExample')

<unknown id>
AC12345
Sequência Exemplo
{'evidence': 'Sequência simulada', 'organism': 'Synthetic construct'}


In [51]:
''' Pegando um objeto SeqRecord na memória e transformando em uma string exatamente no formato de um arquivo '''

record = SeqRecord(
    Seq(
        'MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD'
        'GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK'
        'NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM'
        'SSAC'
    ),
    id='gi|14150838|gb|AAK54648.1|AF376133_1',
    description='chalcone synthase [Cucumis sativus]',
)

# pega um objeto SeqRecord na memória e
# transforma em uma string exatamente no formato de um arquivo

print(record.format('fasta'))

>gi|14150838|gb|AAK54648.1|AF376133_1 chalcone synthase [Cucumis sativus]
MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD
GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK
NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM
SSAC



In [52]:
''' Modificando os dados '''

# Modificando o ID da sequência
record_iterator = SeqIO.parse('ls_orchid.fasta', 'fasta')
first_record = next(record_iterator)

print(first_record.id)  # original

# para mudar:
# id é o atributo do objeto SeqRecord, isso funcionada para qualquer um deles
first_record.id = 'new_id'
print(first_record.id)  # objeto.atributo = novo_valor

# Ajustando ID + Description corretamente
first_record.description = first_record.id + ' ' + 'desired new description'
print(first_record.format('fasta')[:200])

gi|2765658|emb|Z78533.1|CIZ78533
new_id
>new_id desired new description
CGTAACAAGGTTTCCGTAGGTGAACCTGCGGAAGGATCATTGATGAGACCGTGGAATAAA
CGATCGAGTGAATCCGGAGGACCGGTGTACTCAGCTCACCGGGGGCATTGCTCCCGTGGT
GACCCTGATTTGTTGTTGGGCCGCCTCGGGAGCGTCCATGGCGGGT


#### Fatiando um SeqRecord


In [53]:
''' Segmentando um registro GenBank '''

record = SeqIO.read('NC_005816.gb', 'genbank')
print(len(record))  # 9609 bases
print(len(record.features))  # 41 features

# Descobrindo a posição do gene e CDS de interesse
print(record.features[20])
print(record.features[21])

# Segmentando

sub_record = record[4300:4800]
print(len(sub_record))  # 500 bases
print(len(sub_record.features))  # 2 features

# não apenas corto a sequência, mas ajusto as posições das features,
# manteve apenas as features que estão totalmente dentro do trecho
# e removeu anotações gerais que podem não fazer mais sentido
# mas mantém a descrição, então para corrigir isso:

new_description = 'Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, partial'

sub_record.description = new_description
print(sub_record.description)

# Formatando para GenBank
print(sub_record.format('genbank')[:200])

9609
41
type: gene
location: [4342:4780](+)
qualifiers:
    Key: db_xref, Value: ['GeneID:2767712']
    Key: gene, Value: ['pim']
    Key: locus_tag, Value: ['YP_pPCP05']

type: CDS
location: [4342:4780](+)
qualifiers:
    Key: codon_start, Value: ['1']
    Key: db_xref, Value: ['GI:45478716', 'GeneID:2767712']
    Key: gene, Value: ['pim']
    Key: locus_tag, Value: ['YP_pPCP05']
    Key: note, Value: ['similar to many previously sequenced pesticin immunity protein entries of Yersinia pestis plasmid pPCP, e.g. gi| 16082683|,ref|NP_395230.1| (NC_003132) , gi|1200166|emb|CAA90861.1| (Z54145 ) , gi|1488655| emb|CAA63439.1| (X92856) , gi|2996219|gb|AAC62543.1| (AF053945) , and gi|5763814|emb|CAB531 67.1| (AL109969)']
    Key: product, Value: ['pesticin immunity protein']
    Key: protein_id, Value: ['NP_995571.1']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MGGGMISKLFCLALIFLSSSGLAEKNTYTAKDILQNLELNTFGNSLSHGIYGKQTTFKQTEFTNIKSNTKKHIALINKDNSWMISLKILGIKRDEYTVCFEDFSLIRPP

#### Escrevendo arquivos


In [54]:
import os

''' Escrita de arquivos de sequência '''

# O SeqIO.parse() é para LER arquivos de sequência,
# O SeqIO.write() é para ESCREVER arquivos de sequência.

# Estrutura da função: SeqIO.write(registros, arquivo_saida, formato)

# SeqRecord e Seq foram previamente importados
rec1 = SeqRecord(Seq('ATGCGTACGTAG'), id='ID1',
                 description='sequencia exemplo')
rec2 = SeqRecord(Seq('ATGGGCTTTAAA'), id='ID2', description='outra sequencia')
rec3 = SeqRecord(Seq('ATGAAATTTGGG'), id='ID3', description='mais uma')

my_records = [rec1, rec2, rec3]
n = SeqIO.write(my_records, 'my_example.fna', 'fasta')
print(n)  # mostra quantos registros foram escritos, importante quando vem de um iterador
# Se o arquivo já existir, ele vai sobrescrever, para evitar isso:


registros = [rec1, rec2, rec3]

if not os.path.exists('saida.fasta'):
    n = SeqIO.write(registros, 'saida.fasta', 'fasta')
    print(f'{n} registros foram gravados.')
else:
    print('O arquivo já existe e não foi sobrescrito.')

3
O arquivo já existe e não foi sobrescrito.


### Seq Feature


In [55]:
from Bio.SeqFeature import SeqFeature, SimpleLocation
from Bio.SeqFeature import AfterPosition, BetweenPosition

''' Localização de features '''

# isso quer dizer que existe um gene da posição 5 até 12
feature = SeqFeature(SimpleLocation(5, 12), type='gene')
# importante lembrar que o 12 não é incluído

# Fita reversa
feature = SeqFeature(SimpleLocation(5, 12, strand=-1), type='gene')
# o gene está nessas posições, mas na fita complementar, parecido com reverse complement

# localizando uma feature
# feature começa depois da posição 5, mas não sabemos exatamente onde
start_pos = AfterPosition(5)
end_pos = BetweenPosition(9, left=8, right=9)  # A feature termina entre 8 e 9

location = SimpleLocation(start_pos, end_pos)
feature = SeqFeature(location, type='gene')

print(location)
print(feature)

[>5:(8^9)]
type: gene
location: [>5:(8^9)]
qualifiers:



In [56]:
from Bio.SeqFeature import CompoundLocation

''' Localizando SNP '''

seq = Seq('ATGCGTACGTAGCTAGCTAGCGTAGCTAGCTGACTGATCGTAGCTAGCTAGCTAG')
record = SeqRecord(seq, id='Seq_teste', description='Sequência exemplo')

# Gene com DOIS éxons (simulando splicing)
exon1 = SimpleLocation(5, 15)     # primeiro éxon: bases 5–14
exon2 = SimpleLocation(25, 35)    # segundo éxon: bases 25–34

gene_location = CompoundLocation([exon1, exon2])  # junção dos dois éxons
gene_feature = SeqFeature(gene_location, type='gene',
                          qualifiers={'gene': ['geneA']})

# Região codificante (CDS) dentro do gene (mesma localização aqui, simplificado)
cds_feature = SeqFeature(gene_location, type='CDS', qualifiers={
                         'product': ['Proteína X']})

# Região qualquer fora do gene
promoter = SeqFeature(SimpleLocation(0, 4), type='promoter')

# Adicionando as features ao registro
record.features = [promoter, gene_feature, cds_feature]

# Definindo um SNP
my_snp = 28  # posição do SNP (contagem Python começa em 0)
print('SNP na posição:', my_snp)

# Teste de localização
print('O SNP está dentro de quais features?')

for feature in record.features:
    if my_snp in feature:
        print('SNP está dentro de:', feature.type)
        print('\tQualificadores:', feature.qualifiers)

SNP na posição: 28
O SNP está dentro de quais features?
SNP está dentro de: gene
	Qualificadores: {'gene': ['geneA']}
SNP está dentro de: CDS
	Qualificadores: {'product': ['Proteína X']}


#### Exemplo real do Seq Feature em um arquivo GenBank


In [57]:
''' Localizando SNP em um arquivo GenBank real '''

my_snp = 4350
record = SeqIO.read('NC_005816.gb', 'genbank')
for feature in record.features:
    if my_snp in feature:
        print(feature.type, feature.qualifiers.get('db_xref'))

source ['taxon:229193']
gene ['GeneID:2767712']
CDS ['GI:45478716', 'GeneID:2767712']


## Bio Align


Um alinhamento de sequências é um conjunto de duas ou mais sequências organizadas de forma que:

- Tenham o mesmo comprimento
- Possam conter lacunas ("-"), casamento exato ("|") e _mismatch_ (".")
- Permitam comparar posições equivalentes


In [58]:
from Bio.Align import Alignment

''' Alinhamentos de sequências '''

# Exemplo básico:
# from Bio import AlignIO
# alignment = AlignIO.read('exemplo.aln', 'clustal')
# print(alignment)

# Criando um alinhamento na mão (conceitual)
seq1 = SeqRecord(Seq('ATGCTAGC'), id='Seq1')
seq2 = SeqRecord(Seq('ATG-TAGC'), id='Seq2')

alignment = Alignment([seq1, seq2])
print('Alinhamento:')
print(alignment)

# Propriedades importantes
print('Tamanho:', len(alignment))  # número de sequências
# (Número de Sequências, Quantidade de nucleotídeos alinhados)
print('Shape:', alignment.shape)

Alinhamento:
Seq1              0 ATGCTAGC 8
                  0 |||-|||| 8
Seq2              0 ATG-TAGC 8

Tamanho: 2
Shape: (2, 8)


### Criando um Alignment a partir de sequências já alinhadas


In [59]:
''' Alinhamento a partir de sequências com lacunas '''

# Considerando as 3 sequências à seguir:
# 1. CGGTTTTT
# 2. AG-TTT--
# 3. AGGTTT--
# O objeto Alignment não guarda os traços diretamente, ele guarda as sequências sem lacunas

# Para resolver isso, convertemos as linhas para bytes
lines = ['CGGTTTTT', 'AG-TTT--', 'AGGTTT--']

# O Alignment requere que os valores sejam em bytes
lines = [line.encode() for line in lines]
print('Linhas em bytes:\t', lines)

# Extrair sequências reais + coordenadas
sequences, coordinates = Alignment.parse_printed_alignment(lines)
print('Sequências:\t\t', sequences, '\n')
print('Coordenadas:')
print(coordinates, '\n')

# Voltando para string
sequences = [seq.decode() for seq in sequences]
print('Sequências decodificadas:', sequences)
# Os números do resultado dizem quais trechos de cada sequência participam do alinhamento

# pre_seq_0 = 'C'
# sequences[0] = pre_seq_0 + sequences[0]
sequences[0] = 'C' + sequences[0]
sequences[1] = sequences[1] + 'AA'
coordinates[0, :] += 1
# coordinates[0, :] += len(pre_seq_0)

alignment = Alignment(sequences, coordinates)
print('Alinhamento:')
print(alignment)

Linhas em bytes:	 [b'CGGTTTTT', b'AG-TTT--', b'AGGTTT--']
Sequências:		 [b'CGGTTTTT', b'AGTTT', b'AGGTTT'] 

Coordenadas:
[[0 2 3 6 8]
 [0 2 2 5 5]
 [0 2 3 6 6]] 

Sequências decodificadas: ['CGGTTTTT', 'AGTTT', 'AGGTTT']
Alinhamento:
                  1 CGGTTTTT 9
                  0 AG-TTT-- 5
                  0 AGGTTT-- 6

 [b'CGGTTTTT', b'AG-TTT--', b'AGGTTT--']
Sequências:		 [b'CGGTTTTT', b'AGTTT', b'AGGTTT'] 

Coordenadas:
[[0 2 3 6 8]
 [0 2 2 5 5]
 [0 2 3 6 6]] 

Sequências decodificadas: ['CGGTTTTT', 'AGTTT', 'AGGTTT']
Alinhamento:
                  1 CGGTTTTT 9
                  0 AG-TTT-- 5
                  0 AGGTTT-- 6



### Comparando alinhamentos


In [60]:
''' Alinhamento por pares '''

alignment = Alignment(sequences, coordinates)
print('Alinhamento')
print(alignment)

pairwise_alignment = alignment[:2, :]
# Mostra quais trechos das sequências foram alinhados entre si (ignorando lacunas).
print('Alinhamento por pares:')
print(pairwise_alignment)

print('Blocos de alinhamentos:')
print(pairwise_alignment.aligned)
# o primeiro bloco do resultado é a sequência alvo (target), sendo esta a primeira dentre as sequências.

# Então, para vermos cada um dos intervalos, podemos iterar sobre os blocos de alinhamento
# e extrair os intervalos correspondentes das sequências. Por exemplo:

# Sequências do alinhamento por pares
seq1 = pairwise_alignment.sequences[0]
seq2 = pairwise_alignment.sequences[1]

# Blocos alinhados (intervalos sem gaps)
blocos_seq1 = pairwise_alignment.aligned[0]
blocos_seq2 = pairwise_alignment.aligned[1]

# Percorre cada bloco alinhado
for i in range(len(blocos_seq1)):
    inicio1, fim1 = blocos_seq1[i]
    inicio2, fim2 = blocos_seq2[i]

    trecho1 = seq1[inicio1:fim1]
    trecho2 = seq2[inicio2:fim2]

    print(f'Bloco {i + 1}')
    print('Seq 1:', trecho1)
    print('Seq 2:', trecho2)

# Por fim, entende-se que o aligned nos informa quais são os intervalos de
# alinhamento entre as diferentes bases nitrogenadas nas várias sequências,
# onde, para cada lista interna, vemos um par de números, sendo estes o início e fim do alinhamento.

Alinhamento
                  1 CGGTTTTT 9
                  0 AG-TTT-- 5
                  0 AGGTTT-- 6

Alinhamento por pares:
target            1 CGGTTTTT 9
                  0 .|-|||-- 8
query             0 AG-TTT-- 5

Blocos de alinhamentos:
[[[1 3]
  [4 7]]

 [[0 2]
  [2 5]]]
Bloco 1
Seq 1: CG
Seq 2: AG
Bloco 2
Seq 1: TTT
Seq 2: TTT


### Contagem de identidades, mismatches, lacunas e estatísticas


In [61]:
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices

''' Alinhamento simples (DNA) e contagem básica '''

# Calculando estatísticas básicas do alinhamento
counts = pairwise_alignment.counts()

# O objeto AlignmentCounts resume o alinhamento
print(counts)

# Acessando propriedades individualmente
print('Letras alinhadas:', counts.aligned)
print('Identidades:', counts.identities)
print('Mismatches:', counts.mismatches)
print('Lacunas (gaps):', counts.gaps)
print('Inserções:', counts.insertions)
print('Deleções:', counts.deletions)

AlignmentCounts object with
    aligned = 5:
        identities = 4,
        mismatches = 1.
    gaps = 3:
        left_gaps = 0:
            left_insertions = 0:
                open_left_insertions = 0,
                extend_left_insertions = 0;
            left_deletions = 0:
                open_left_deletions = 0,
                extend_left_deletions = 0;
        internal_gaps = 1:
            internal_insertions = 0:
                open_internal_insertions = 0,
                extend_internal_insertions = 0;
            internal_deletions = 1:
                open_internal_deletions = 1,
                extend_internal_deletions = 0;
        right_gaps = 2:
            right_insertions = 0:
                open_right_insertions = 0,
                extend_right_insertions = 0;
            right_deletions = 2:
                open_right_deletions = 1,
                extend_right_deletions = 1.

Letras alinhadas: 5
Identidades: 4
Mismatches: 1
Lacunas (gaps): 3
Inserções: 0
Del

In [62]:
''' Alinhamento simples (DNA) e contagem com wildcard '''

# Uso de caractere wildcard (*, ?)

# Substituindo um nucleotídeo por um caractere curinga '?'
pairwise_alignment.sequences[0] = 'CCGGT?TTT'
print(pairwise_alignment)

# Contagem padrão (sem definir curinga)
counts = pairwise_alignment.counts()
print('Identidades (sem wildcard):', counts.identities)
print('Mismatches (sem wildcard):', counts.mismatches)

# Contagem ignorando o caractere '?'
counts = pairwise_alignment.counts('?')
print('Identidades (com wildcard):', counts.identities)
print('Mismatches (com wildcard):', counts.mismatches)

target            1 CGGT?TTT 9
                  0 .|-|.|-- 8
query             0 AG-TTT-- 5

Identidades (sem wildcard): 3
Mismatches (sem wildcard): 2
Identidades (com wildcard): 3
Mismatches (com wildcard): 1


In [63]:
''' Alinhamento de proteínas e matrizes de substituição '''

# Criando um alinhamento de proteínas
protein_alignment = Alignment(
    [
        'EPQSDPSVEPPLSQETFSDLWKLLPE',
        'EPSSETGMDPPLSQETFEDLWSLLPD'
    ]
)

print(protein_alignment)

# Contagem sem matriz de substituição
counts = protein_alignment.counts()
print('Identidades:', counts.identities)
print('Mismatches:', counts.mismatches)
print('Positivos:', counts.positives)              # None
print('Substitution score:', counts.substitution_score)  # None

# Carregando matriz BLOSUM62
blosum62 = substitution_matrices.load('BLOSUM62')

# Contagem usando a matriz BLOSUM62
counts = protein_alignment.counts(blosum62)
print('Contagem da Matriz BLOSUM62')
print('Identidades:', counts.identities)
print('Mismatches:', counts.mismatches)
print('Positivos:', counts.positives)
print('Score de substituição:', counts.substitution_score)

# Contagem usando a matriz BLOSUM45
blosum45 = substitution_matrices.load('BLOSUM45')
counts = protein_alignment.counts(blosum45)
print('Contagem da Matriz BLOSUM45')
print('Identidades:', counts.identities)
print('Mismatches:', counts.mismatches)
print('Positivos:', counts.positives)
print('Score de substituição:', counts.substitution_score)
# Dá pra fazer o mesmo com blosum90, só substituir o número por 90

target            0 EPQSDPSVEPPLSQETFSDLWKLLPE 26
                  0 ||.|.....||||||||.|||.|||. 26
query             0 EPSSETGMDPPLSQETFEDLWSLLPD 26

Identidades: 17
Mismatches: 9
Positivos: None
Substitution score: None
Contagem da Matriz BLOSUM62
Identidades: 17
Mismatches: 9
Positivos: 21
Score de substituição: 101.0
Contagem da Matriz BLOSUM45
Identidades: 17
Mismatches: 9
Positivos: 21
Score de substituição: 122.0


In [64]:
''' Alinhamento de proteínas e contagem com gaps '''

# Usando PairwiseAligner para score total (substituição + gaps)
pairwise_alignment.sequences[0] = 'CCGGTTTTT'
print(pairwise_alignment)

# Criando um alinhador no modo BLASTN (DNA)
aligner = PairwiseAligner('blastn')

# Contagem incluindo pontuação de gaps
counts = pairwise_alignment.counts(aligner)

print(counts)
print('Score total:', counts.score)
print('Score de substituição:', counts.substitution_score)
print('Score de gaps:', counts.gap_score)


# Alinhador para proteínas (BLASTP)
aligner = PairwiseAligner('blastp')
counts = protein_alignment.counts(aligner)

print(counts)
print('Score total:', counts.score)
print('Score de substituição:', counts.substitution_score)
print('Score de gaps:', counts.gap_score)

target            1 CGGTTTTT 9
                  0 .|-|||-- 8
query             0 AG-TTT-- 5

AlignmentCounts object with
    score = -11.0:
        substitution_score = 5.0,
        gap_score = -16.0.
    aligned = 5:
        identities = 4,
        positives = 4,
        mismatches = 1.
    gaps = 3:
        left_gaps = 0:
            left_insertions = 0:
                open_left_insertions = 0,
                extend_left_insertions = 0;
            left_deletions = 0:
                open_left_deletions = 0,
                extend_left_deletions = 0;
        internal_gaps = 1:
            internal_insertions = 0:
                open_internal_insertions = 0,
                extend_internal_insertions = 0;
            internal_deletions = 1:
                open_internal_deletions = 1,
                extend_internal_deletions = 0;
        right_gaps = 2:
            right_insertions = 0:
                open_right_insertions = 0,
                extend_right_insertions = 0;
      

### Alinhamento Múltiplo


In [65]:
''' Criando um alinhamento com três sequências '''

print(alignment)

# Contagem somada para todos os pares de sequências

counts = alignment.counts()

print('Letras alinhadas:', counts.aligned)
print('Identidades:', counts.identities)
print('Mismatches:', counts.mismatches)
print('Inserções:', counts.insertions)
print('Deleções:', counts.deletions)
print('Total de gaps:', counts.gaps)
print('Gaps à esquerda:', counts.left_gaps)
print('Gaps internos:', counts.internal_gaps)
print('Gaps à direita:', counts.right_gaps)

# Letter frequencie: conta quantas vezes cada letra aparece em cada coluna

# Cada chave: uma letra; cada array: colunas do alinhamento
print(alignment.frequencies)

# Quantidade de substituições:
substituicao = alignment.substitutions
print(substituicao)

                  1 CGGTTTTT 9
                  0 AG-TTT-- 5
                  0 AGGTTT-- 6

Letras alinhadas: 16
Identidades: 14
Mismatches: 2
Inserções: 1
Deleções: 5
Total de gaps: 6
Gaps à esquerda: 0
Gaps internos: 2
Gaps à direita: 4
{'C': array([1., 0., 0., 0., 0., 0., 0., 0.]), 'G': array([0., 3., 2., 0., 0., 0., 0., 0.]), 'T': array([0., 0., 0., 3., 3., 3., 1., 1.]), 'A': array([2., 0., 0., 0., 0., 0., 0., 0.]), '-': array([0., 0., 1., 0., 0., 0., 2., 2.])}
    A   C   G   T
A 1.0 0.0 0.0 0.0
C 2.0 0.0 0.0 0.0
G 0.0 0.0 4.0 0.0
T 0.0 0.0 0.0 9.0



In [66]:
import numpy as np

''' Adicionando alinhamentos '''


# Seq e SeqRecord foram previamente importados
a1 = SeqRecord(Seq('AAAAC'), id='Alpha')
b1 = SeqRecord(Seq('AAAC'), id='Beta')
c1 = SeqRecord(Seq('AAAAG'), id='Gamma')
a2 = SeqRecord(Seq('GTT'), id='Alpha')
b2 = SeqRecord(Seq('TT'), id='Beta')
c2 = SeqRecord(Seq('GT'), id='Gamma')
left = Alignment(
    [a1, b1, c1], coordinates=np.array(
        [[0, 3, 4, 5], [0, 3, 3, 4], [0, 3, 4, 5]])
)
left.annotations = {'tool': 'demo', 'name': 'start'}
left.column_annotations = {'stats': 'CCCXC'}
right = Alignment(
    [a2, b2, c2], coordinates=np.array(
        [[0, 1, 2, 3], [0, 0, 1, 2], [0, 1, 1, 2]])
)
right.annotations = {'tool': 'demo', 'name': 'end'}
right.column_annotations = {'stats': 'CXC'}

print(left)
print(right)

combined = left + right
print(combined)

# Os alinhamentos podem ser combinados para formar
# um alinhamento estendido se tiverem o mesmo número de linhas.

Alpha             0 AAAAC 5
Beta              0 AAA-C 4
Gamma             0 AAAAG 5

Alpha             0 GTT 3
Beta              0 -TT 2
Gamma             0 G-T 2

Alpha             0 AAAACGTT 8
Beta              0 AAA-C-TT 6
Gamma             0 AAAAGG-T 7



In [67]:
''' Mapeamento de um alinhamento de sequências aos pares '''

# 1- Alinhamento par a par de um transcrito com um cromossomo:
chromosome = 'AAAAAAAACCCCCCCAAAAAAAAAAAGGGGGGAAAAAAAA'
transcript = 'CCCCCCCGGGGGG'  # com o intron removido
sequences1 = [chromosome, transcript]

coordinates1 = np.array([
    [8, 15, 26, 32],   # cromossomo
    [0, 7, 7, 13]      # transcrito
])

alignment1 = Alignment(sequences1, coordinates1)
print(alignment1)

# 2- Alinhamento par a par entre o transcrito e uma sequência:
rnaseq = 'CCCCGGGG'
sequences2 = [transcript, rnaseq]
coordinates2 = np.array([
    [3, 11],   # transcrito
    [0, 8]     # RNAseq
])
alignment2 = Alignment(sequences2, coordinates2)
print(alignment2)

# Método .map(): nesse exemplo faz a interseção de coordenadas
# apenas as coordenadas de alignment1 e alignment2
# são usadas para construir as coordenadas de alignment3
alignment3 = alignment1.map(alignment2)
print(alignment3)
print(alignment3.coordinates)

# Para alinhamento múltiplo é só usar o .mapall()

target            8 CCCCCCCAAAAAAAAAAAGGGGGG 32
                  0 |||||||-----------|||||| 24
query             0 CCCCCCC-----------GGGGGG 13

target            3 CCCCGGGG 11
                  0 ||||||||  8
query             0 CCCCGGGG  8

target           11 CCCCAAAAAAAAAAAGGGG 30
                  0 ||||-----------|||| 19
query             0 CCCC-----------GGGG  8

[[11 15 26 30]
 [ 0  4  4  8]]


In [68]:
from Bio import Align

''' Leitura de Alinhamento '''


# 1) LENDO UM ARQUIVO COM MÚLTIPLOS ALINHAMENTOS (MAF)

# parse() retorna um ITERADOR (não carrega tudo na memória)
alignments = Align.parse("ucsc_mm9_chr10.maf", "maf")

print(type(alignments))

# 2) ACESSANDO METADADOS DO ARQUIVO

# Alguns formatos (como MAF) guardam metadados
print("\nMetadata do arquivo:")
print(alignments.metadata)

# 3) ITERANDO SOBRE OS ALINHAMENTOS

print("\nNúmero de sequências em cada bloco de alinhamento:")

for alignment in alignments:
    print(len(alignment.sequences))

# 4) SABENDO QUANTOS ALINHAMENTOS EXISTEM

# Pode ser lento na primeira vez (se tiver que contar)
print("\nTotal de alinhamentos:")
print(len(alignments))


# 5) CONVERTENDO PARA LISTA (PERDE METADATA)

alignment_list = list(alignments)

print("\nTipo após converter para lista:")
print(type(alignment_list))

print("\nExemplo de alinhamento da lista:")
print(alignment_list[0])


# 6) FORMA CORRETA: MANTER METADATA

# Recarregar iterator
alignments = Align.parse("ucsc_mm9_chr10.maf", "maf")

# Converte mantendo metadata
alignments = alignments[:]

print("\nTipo após slicing [:]:")
print(type(alignments))

print("\nMetadata preservado:")
print(alignments.metadata)

print("\nExemplo alinhamento:")
print(alignments[0])


# 7) ESCREVENDO ALINHAMENTOS EM OUTRO FORMATO

target_file = "saida_clustal.aln"

Align.write(alignments, target_file, "clustal")

print("\nAlinhamentos escritos em formato Clustal.")

# 8) ESCREVENDO COM METADATA CUSTOMIZADO

alignments_with_meta = Align.Alignments(alignments)

alignments_with_meta.metadata = {
    "Program": "Biopython",
    "Version": "1.81"
}

Align.write(alignments_with_meta, "saida_com_metadata.aln", "clustal")

print("\nAlinhamentos escritos com metadata customizado.")

<class 'Bio.Align.maf.AlignmentIterator'>

Metadata do arquivo:
{'MAF Version': '1', 'Scoring': 'autoMZ.v1'}

Número de sequências em cada bloco de alinhamento:
2
4
5
6
7
7
1
4
1
5
6
4
1
9
10
7
1
3
1
4
3
5
1
2
2
1
2
3
6
5
1
2
7
10
1
4
1
9
10
11
12
13
14
15
15
14
7
6

Total de alinhamentos:
48

Tipo após converter para lista:
<class 'list'>

Exemplo de alinhamento da lista:
mm9.chr10   3009319 TCATAGGTATTTATTTTTAAATATGGTTTGCTTTATGGCTAGAACACACCGATTACTTAA
                  0 |||.||.||||||.|.||||||||||||||.|.|||||.||.....||..|.||||||..
oryCun1.s     11087 TCACAGATATTTACTATTAAATATGGTTTGTTATATGGTTACGGTTCATAGGTTACTTGG

mm9.chr10   3009379 AATAGGATTAACC--CCCATACACTTTAAAAATGATTAAACAACATTTCTGCTGCTCGCT
                 60 |||.|||||||||--|..||.||.|..|.||.||.|||.||....||..||....|.|||
oryCun1.s     11147 AATTGGATTAACCTTCTTATTCATTGCAGAATTGGTTACACTGTGTTCTTGACCTTTGCT

mm9.chr10   3009437 CACATTCTTCATAGAAGATGACATAATGTATTTTCCTTTTGGTT 3009481
                120 ....||||.|||.|||..|||..|.|..||.|||||.||||||